# Trees, forests, and boosting

**P1 Core · D2 Independent · 100 minutes**

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/predictive-maintenance/observations.csv', 'observations.csv')
data = pd.read_csv(path)
features = ['Type','Air temperature [K]','Process temperature [K]',
            'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
X_train, X_test, y_train, y_test = train_test_split(
    data[features], data['Machine failure'], test_size=.25, random_state=29, stratify=data['Machine failure'])
pre = ColumnTransformer([('type', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Type'])],
                        remainder='passthrough')

## Task

Fit a controlled tree, OOB random forest, and histogram boosting model under fixed budgets. Return protected-test PR-AUC.

In [ ]:
def compare_models():
    estimators = {
        'tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight='balanced', random_state=29),
        'forest': RandomForestClassifier(n_estimators=120, min_samples_leaf=5, class_weight='balanced',
                                         oob_score=True, n_jobs=1, random_state=29),
        'boosting': HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15, learning_rate=.08,
                                                   class_weight='balanced', random_state=29),
    }
    scores = {}
    fitted = {}
    for name, estimator in estimators.items():
        pipeline = Pipeline([('preprocess', pre), ('model', estimator)]).fit(X_train, y_train)
        scores[name] = average_precision_score(y_test, pipeline.predict_proba(X_test)[:,1])
        fitted[name] = pipeline
    return scores, fitted

In [ ]:
scores, fitted = compare_models()
dummy = DummyClassifier(strategy='prior').fit(X_train, y_train)
dummy_ap = average_precision_score(y_test, dummy.predict_proba(X_test)[:,1])
assert max(scores.values()) > dummy_ap
assert 0 < fitted['forest'].named_steps['model'].oob_score_ < 1
{'dummy': dummy_ap, **scores}

## Transfer

Perturb the sample seed, compare tree/importance instability, relate OOB to test evidence, and explain why predictive importance is non-causal.